# LangGraph & the Unified Pipeline — Code Guide

**Target file:** `agents/pipeline_graph.py`
**Owner:** Jack (orchestration)

> This notebook mirrors `agents/pipeline_graph.py` block by block and teaches the LangGraph concepts our whole codebase is built on. It **runs top to bottom** — every cell just *defines* the graph pieces; nothing calls `run()`, so no agents are constructed and no network/GPU work fires. To actually execute the pipeline, call `run()` in a fresh cell at the end (needs the real agents + data). Its two jobs: (1) teach the LangGraph ideas in plain English, and (2) let the team read the retune loop as one graph instead of tracing Python control flow in their head.

## Why this notebook exists

Every one of our five agents is a LangGraph, and the thing that ties them together — the **retune loop** — is *also* a LangGraph. If you understand the handful of LangGraph ideas below, every agent module in this repo reads the same way. So this guide front-loads the concepts, then walks the actual orchestration file.


## Part A — LangGraph in six ideas

LangGraph lets you describe a program as a **graph**: boxes (nodes) with arrows (edges) and a shared clipboard (state) that flows through them. That sounds heavy, but for us it's just six ideas.

**1. State — the shared clipboard.**
One `TypedDict` that every node can read and write. It is the *only* thing passed between nodes. In this repo the pipeline state is tiny on purpose (file paths + a loop counter) — the real data lives in the CSV/JSON contract files the agents read and write, not in the state.

**2. Node — a plain function `state → partial update`.**
A node takes the whole state and returns a **dict of only the keys it changed**. It does *not* mutate the state in place and it does *not* return the whole thing. LangGraph takes that partial dict and merges it in for you.

**3. Merging & reducers — how updates combine.**
By default a returned key **overwrites** the old value (latest-wins). But you can annotate a state key with a *reducer* so updates **combine** instead — e.g. `Annotated[list, operator.add]` means "append to the list rather than replace it." That is how an agent remembers history across loop passes (you'll see this in the Manager guide).

**4. Edges — the arrows.**
`add_edge("a", "b")` means "after node `a`, always go to node `b`." Two special nodes, `START` and `END`, mark where the graph begins and finishes.

**5. Conditional edges — branching and CYCLES.**
`add_conditional_edges("gate", route, {...})` runs a **router function** that looks at the state and returns a label; the label picks the next node. Crucially, a conditional edge can point **back to an earlier node** — that's a loop. State-carrying cycles are LangGraph's headline feature and the whole reason this file exists.

**6. compile + checkpointer — freeze the shape, remember between runs.**
`builder.compile(checkpointer=...)` turns the description into a runnable graph. A **checkpointer** (e.g. `MemorySaver`) saves the state under a `thread_id` so it survives *between* `.invoke()` calls — that's how the Manager keeps its iteration counter across passes. `recursion_limit` caps how many node-steps one `.invoke()` may take, so a runaway cycle aborts instead of spinning forever.

Keep those six in mind and the rest of this notebook is just reading.


## 1. Module docstring — the design decision

The docstring is worth reading in full because it explains the single most important architectural choice: **compose, don't flatten.**

Each agent is already its own graph with its own internal state keys — and some of those keys *collide* across agents (Sabina and Freddi both use `output_path`). If we flattened all five agents into one big node-set, those keys would fight over one namespace and we'd lose the per-agent test isolation the team built on purpose.

So instead: **each node here just invokes an agent** (calls its `.run()`), and the unified graph only carries the loop's control state — file paths, the loop counter, the gate's verdict. The agents stay black boxes. The retune loop becomes a real graph edge. That was the goal.

The ASCII diagram in the docstring is the map for the whole rest of the file — the `gate → classify` back-arrow is the cycle.


In [ ]:
"""Unified pipeline graph (Increment 2) — the retune loop as a real LangGraph cycle.

`main.py` originally drove the five agents with a plain Python `for`-loop. This
module expresses the *same* flow as ONE compiled LangGraph whose retune loop is a
genuine graph **cycle** (`gate → classify → evaluate → gate`) rather than Python
control flow. That is the piece the earlier design was missing: state-carrying
cycles are LangGraph's headline feature.

## Design choice: compose, don't flatten

Each agent is already its own small graph with its own internal state keys (and
some of those keys collide across agents — e.g. Sabina and Freddi both use
`output_path`). Flattening all five into one node-set would force those keys to
share one namespace and would delete the per-agent test isolation the team built
on purpose. So instead each **node here invokes an agent** (its `.run()` / sub-graph)
and the unified graph only carries the loop's control state — file paths, the loop
counter, and the gate's verdict. The agents stay black boxes; nothing about them
changes. The retune loop becomes a real graph edge; that was the goal.

## Graph shape

    START → process → classify → evaluate → gate ─(retune)→ classify   [CYCLE]
                                              └(proceed)→ explain → finalize → END

`gate` is the Manager: calling it writes either `retune_request.json` (retune) or
`sample_for_explanation.csv` (proceed). Its own checkpointer carries the iteration
counter, accuracy history, and adaptive-param state across cycle passes (Increment
1), so convergence/adaptation work exactly as when driven by the Python loop.

## Testing

`build_pipeline(agents=...)` takes an optional `Agents` bundle so tests can inject
lightweight fakes for the network/GPU-bound agents (Aurora, Nadi) and exercise the
real cycle + Manager gate offline. `Agents.build()` constructs the real ones.
"""

## 2. Imports & the contract file paths

Two things here:

- **The `try/except ModuleNotFoundError` import dance** appears in every agent. It lets the file work both as a package import (`from agents.jack_manager import ...`, how tests load it) *and* as a bare script with `agents/` on the path. Same import, two entry styles.
- **The module-level path constants** (`CODE`, `PREDS`, `EVAL`, `SAMPLE`, `RETUNE`, `EXPL`) are the contract files the nodes hand to each other. They all live under the Manager's `OUTPUT_DIR` except `processed_data.csv`, whose path Aurora returns at runtime. Defining them once here means no node hardcodes a filename.
- **`RECURSION_LIMIT = 60`** is deliberate headroom. One retune cycle is 3 nodes; ~5 iterations plus the tail stays well under 60. We raise it above LangGraph's default of 25 so a legitimate long run isn't mistaken for a runaway loop.


In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass

from typing_extensions import TypedDict

try:
    from agents.aurora_processing import ProcessingAgent
    from agents.nadi_classifier import ClassifierAgent
    from agents.sabina_evaluator import EvaluatorAgent
    from agents.freddi_explanation import ExplanationAgent
    from agents.jack_manager import ManagerAgent, OUTPUT_DIR as OUT
except ModuleNotFoundError:  # running as a bare script with agents/ on sys.path
    from aurora_processing import ProcessingAgent
    from nadi_classifier import ClassifierAgent
    from sabina_evaluator import EvaluatorAgent
    from freddi_explanation import ExplanationAgent
    from jack_manager import ManagerAgent, OUTPUT_DIR as OUT

# Contract files exchanged between the nodes. All live under the Manager's
# OUTPUT_DIR except processed_data.csv, whose path Aurora returns at runtime.
CODE = os.path.join(OUT, "classifier.py")
PREDS = os.path.join(OUT, "predictions_test.csv")
EVAL = os.path.join(OUT, "evaluation_report.json")
SAMPLE = os.path.join(OUT, "sample_for_explanation.csv")
RETUNE = os.path.join(OUT, "retune_request.json")
EXPL = os.path.join(OUT, "explanations.csv")

# A retune cycle is 3 nodes (classify → evaluate → gate); with the process/explain/
# finalize tail, ~5 iterations stays well under this. LangGraph aborts a runaway
# cycle at recursion_limit, so we set headroom rather than rely on the default 25.
RECURSION_LIMIT = 60

## 3. `PipelineState` — the control clipboard

This is idea **#1** from Part A, made concrete. Notice how *small* it is — four keys, all about routing:

- `processed_data_path` — set by `process`, read by `classify`.
- `retune_request_path` — `None` on the first pass, `RETUNE` on cycle passes. This one key is what makes the classifier escalate its params on a loop-back.
- `final_action` — the gate's verdict, `"retune"` or `"proceed"`; the router reads it to branch.
- `iteration` — the Manager's loop counter, surfaced here just for reporting.

`total=False` means every key is optional (nodes fill them in as the graph runs). The per-row prediction data is **not** here — it's in the contract files. The state only holds what the *graph* needs to route and cycle.


In [ ]:
class PipelineState(TypedDict, total=False):
    """Control state carried around the unified graph. Deliberately small — the
    per-row data lives in the contract CSV/JSON files the agents read and write,
    not in here. Only what the *graph* needs to route and cycle lives in state."""

    processed_data_path: str        # set by `process`, read by `classify`
    retune_request_path: str | None  # None on the first pass; RETUNE on cycle passes
    final_action: str               # the gate's verdict: "retune" | "proceed"
    iteration: int                  # Manager's loop counter (for reporting)

## 4. `Agents` — the five agents, bundled for injection

A small dataclass holding the five agent instances the graph drives. Why bundle them instead of constructing them inside `build_pipeline`? **Testability.** Tests can inject lightweight fakes for the network/GPU-bound agents (Aurora downloads prices, Nadi runs FinBERT) while using the real, offline-capable Manager and Evaluator — and still exercise the *real* cycle and gate.

`Agents.build(...)` is the production path: it constructs the real agents and threads the run's config through to each. Note the Manager gets the real `PREDS` path so it never silently falls back to the mock default when sampling.


In [ ]:
@dataclass
class Agents:
    """The five agents the graph drives. Bundled so tests can inject fakes for the
    network/GPU-bound ones while using the real, offline-capable others."""

    aurora: object
    nadi: object
    sabina: object
    manager: object
    freddi: object

    @classmethod
    def build(cls, *, target_accuracy=0.60, max_iterations=5, patience=2,
              min_delta=0.01, sample_size=300, use_ollama=True) -> "Agents":
        """Construct the real agents wired with the run's config. The Manager gets
        the real PREDS path so it never falls back to the mock default."""
        return cls(
            aurora=ProcessingAgent(),
            nadi=ClassifierAgent(),
            sabina=EvaluatorAgent(output_dir=OUT),
            manager=ManagerAgent(predictions_path=PREDS, target_accuracy=target_accuracy,
                                 max_iterations=max_iterations, patience=patience,
                                 min_delta=min_delta, sample_size=sample_size),
            freddi=ExplanationAgent(use_ollama=use_ollama, output_path=EXPL),
        )

## 5. `build_pipeline` — the node functions

Here's the heart of the file. `build_pipeline` defines each node as a **closure** — the inner functions close over `agents`, `threshold`, and `data_dir`, which is why none of those non-serialisable objects need to live in the graph *state* (idea #1: keep state small and serialisable).

Read each node against the six ideas:

- **`process`** — runs Aurora once, before the loop. Returns `{"processed_data_path": ...}` — a partial update (idea #2).
- **`classify`** — runs Nadi. On a cycle pass it reads `state.get("retune_request_path")`; on the first pass that's `None` (fresh classifier). Returns `{}` because it changes no routing state — it just produces a file downstream nodes read by their constant paths.
- **`evaluate`** — runs Sabina, scores predictions. Also returns `{}`.

> The full `build_pipeline` function is one code cell at the end of this section (5b) — it's a single Python function, so we keep it whole and runnable rather than chopping it mid-body. Read sections 5, 5a, 5b against the one cell below them.


### 5a. `evaluate` and the `gate` node

`gate` is the interesting one — it's the **Manager**. Calling `agents.manager.run(...)` applies the accuracy gate (with convergence + adaptive retune, all covered in the Manager guide) and, as a *side effect*, writes **either** `retune_request.json` (retune) **or** `sample_for_explanation.csv` (proceed).

The node then surfaces the Manager's verdict into the pipeline state — `final_action` and `iteration` — so the router can branch. And when the verdict is retune, it sets `retune_request_path = RETUNE`, which is the exact key `classify` reads on the loop-back. That one line is the wire that carries feedback around the cycle.


### 5b. `explain`, `finalize`, `route` — and the whole function

The remaining nodes:

- **`explain`** — runs Freddi to justify each sampled prediction.
- **`finalize`** — runs the Manager *again*, now with explanations present, so it writes `final_results.csv` and `final_report.json`. This is **not** a new iteration — the gate already decided to proceed.
- **`route`** — this is idea **#5**, the router. It reads `final_action` and returns `"retune"` or `"proceed"`. That returned label is looked up in the mapping passed to `add_conditional_edges` to pick the next node. This tiny function is the branch point of the entire cycle.

The cell below is the **entire `build_pipeline` function** — everything sections 5, 5a, and 5b described, plus the graph wiring (section 6). Defining it runs no nodes; the `from langgraph.graph import ...` inside only fires when the function is actually called.


In [ ]:
def build_pipeline(agents: Agents, *, threshold=0.01, data_dir=None, checkpointer=None):
    """Compile the unified pipeline graph. `agents` supplies the five agents (real
    or fake); `threshold` is Aurora's labelling band; `data_dir` overrides where
    Aurora reads fnspid_raw.csv (defaults to the repo `data/`). The node functions
    close over these, so no non-serialisable objects live in the graph state.
    """
    from langgraph.graph import StateGraph, START, END

    def process(state: PipelineState) -> dict:
        """Aurora: build the labelled dataset. Runs once, before the loop."""
        extra = {"data_dir": data_dir} if data_dir else {}
        processed = agents.aurora.run(threshold=threshold, **extra)["processed_data_path"]
        return {"processed_data_path": processed}

    def classify(state: PipelineState) -> dict:
        """Nadi: (re)generate and run the classifier. On cycle passes,
        `retune_request_path` points at the Manager's latest retune request so the
        params escalate; on the first pass it is None (fresh classifier)."""
        agents.nadi.run(processed_data=state["processed_data_path"],
                        classifier_code=CODE, predictions=PREDS,
                        retune_request=state.get("retune_request_path"))
        return {}

    def evaluate(state: PipelineState) -> dict:
        """Sabina: score the predictions and write evaluation_report.json."""
        agents.sabina.run(predictions=PREDS, classifier_code=CODE)
        return {}

    def gate(state: PipelineState) -> dict:
        """Manager gate. Invoking it applies the accuracy gate (with convergence +
        adaptive retune) and, as a side effect, writes EITHER retune_request.json
        (retune) OR sample_for_explanation.csv (proceed). We surface its verdict so
        the router can branch, and expose RETUNE as the next classify input when
        retuning."""
        st = agents.manager.run(evaluation_report=EVAL)
        out = {"final_action": st["final_action"], "iteration": st["iteration"]}
        if st["final_action"] == "retune":
            out["retune_request_path"] = RETUNE  # fed back into `classify` on the cycle
        return out

    def explain(state: PipelineState) -> dict:
        """Freddi: justify each sampled prediction into explanations.csv."""
        agents.freddi.run(sample_for_explanation=SAMPLE, output=EXPL)
        return {}

    def finalize(state: PipelineState) -> dict:
        """Manager again, now with explanations present → writes final_results.csv
        and final_report.json. Not a new iteration (the gate already proceeded)."""
        st = agents.manager.run(evaluation_report=EVAL, explanations=EXPL)
        return {"iteration": st["iteration"]}

    def route(state: PipelineState) -> str:
        """The cycle's branch point: loop back to `classify` on retune, else move
        on to the explanation stage."""
        return "retune" if state["final_action"] == "retune" else "proceed"

    b = StateGraph(PipelineState)
    for name, fn in [("process", process), ("classify", classify), ("evaluate", evaluate),
                     ("gate", gate), ("explain", explain), ("finalize", finalize)]:
        b.add_node(name, fn)
    b.add_edge(START, "process")
    b.add_edge("process", "classify")
    b.add_edge("classify", "evaluate")
    b.add_edge("evaluate", "gate")
    b.add_conditional_edges("gate", route, {"retune": "classify", "proceed": "explain"})
    b.add_edge("explain", "finalize")
    b.add_edge("finalize", END)
    return b.compile(checkpointer=checkpointer)

## 6. Wiring the graph — `StateGraph`, edges, the cycle

Now the six ideas come together into an actual graph:

1. `StateGraph(PipelineState)` — declare the state schema (idea #1).
2. The loop adds all six nodes (idea #2).
3. `add_edge(START, "process")` … the straight-line edges lay out `process → classify → evaluate → gate` (idea #4).
4. `add_conditional_edges("gate", route, {"retune": "classify", "proceed": "explain"})` — **this is the cycle** (idea #5). `"retune"` points *back* to `classify`, so the graph loops; `"proceed"` moves on to `explain`.
5. `explain → finalize → END` finishes the run.
6. `b.compile(checkpointer=checkpointer)` freezes it into a runnable graph (idea #6).

Compare the wiring block inside the cell above (`b = StateGraph(...)` down to `b.compile(...)`) to the ASCII diagram in the docstring (section 1) — the code is a line-for-line transcription of that picture. It's the tail of the `build_pipeline` cell you just read.


## 7. `run` — the production entry point

`main.py` calls this. It:

1. Builds the real agents via `Agents.build(...)` with the run's config.
2. Compiles the pipeline with a fresh `MemorySaver` checkpointer (idea #6) — so the Manager's iteration counter and history survive across the cycle's passes.
3. Calls `graph.invoke(...)` with the initial state `{"retune_request_path": None}` (first pass = fresh classifier) and a config carrying the `thread_id` and our raised `recursion_limit`.

The return value is the final graph state. Everything the run actually *produced* is on disk in `outputs/` as the contract files.

That's the whole orchestration layer: six nodes, one back-edge, one small state dict. The complexity lives inside the agents; this file just choreographs them.


In [ ]:
def run(*, threshold=0.01, target_accuracy=0.60, max_iterations=5, patience=2,
        min_delta=0.01, sample_size=300, use_ollama=True, data_dir=None) -> dict:
    """Build the pipeline with real agents and run it once end to end, returning the
    final graph state. This is the entry point `main.py` calls."""
    from langgraph.checkpoint.memory import MemorySaver

    agents = Agents.build(target_accuracy=target_accuracy, max_iterations=max_iterations,
                          patience=patience, min_delta=min_delta, sample_size=sample_size,
                          use_ollama=use_ollama)
    graph = build_pipeline(agents, threshold=threshold, data_dir=data_dir,
                           checkpointer=MemorySaver())
    return graph.invoke(
        {"retune_request_path": None},
        {"configurable": {"thread_id": "pipeline"}, "recursion_limit": RECURSION_LIMIT},
    )